# VOX AI ENGINE — Temporary Google Colab XTTS-v2 Inference Node

**Architecture Role:**
`Android APK` $\rightarrow$ `Render Node.js API` $\rightarrow$ `Google Colab XTTS-v2 GPU Engine` $\rightarrow$ `Generated WAV Audio`

**Important Usage Notice:**
- This notebook is strictly for **temporary personal testing and development**.
- Google Colab runtimes automatically disconnect after idle periods or session limits. Do not treat Colab as permanent production hosting.
- Only clone voices that you own or have explicit permission to clone.
- Before running, go to **Runtime $\rightarrow$ Change runtime type $\rightarrow$ T4 GPU**.

In [ ]:
#@title 1. GPU Hardware Verification & Dependencies Installation
import subprocess

print("Checking NVIDIA GPU availability...")
gpu_check = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
if gpu_check.returncode != 0:
    print("[WARNING] No GPU detected! XTTS-v2 will run very slowly on CPU.")
    print("Go to Runtime -> Change runtime type -> Hardware accelerator -> T4 GPU, then re-run.")
else:
    print("[OK] NVIDIA GPU detected:")
    print(gpu_check.stdout.splitlines()[2])

print("\nInstalling maintained Coqui XTTS-v2, FastAPI, Uvicorn, and PyNgrok...")
!pip install -q -U coqui-tts fastapi "uvicorn[standard]" python-multipart pyngrok nest-asyncio soundfile
print("[OK] Dependencies installed successfully.")

In [ ]:
#@title 2. Load Pretrained XTTS-v2 Model & Initialize Voice Vault
import os
import json
import torch
from TTS.api import TTS

os.environ["COQUI_TOS_AGREED"] = "1"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Initializing XTTS-v2 on device: {DEVICE.upper()}...")

MODEL_NAME = "tts_models/multilingual/multi-dataset/xtts_v2"
try:
    tts = TTS(MODEL_NAME).to(DEVICE)
    xtts_model = tts.synthesizer.tts_model
    print(f"[OK] {MODEL_NAME} loaded into {DEVICE.upper()} memory.")
except Exception as e:
    raise RuntimeError(f"Failed to download/load XTTS-v2 model: {e}")

BASE_DIR = "/content/vox_ai_engine"
VOICES_DIR = os.path.join(BASE_DIR, "voices")
OUTPUT_DIR = os.path.join(BASE_DIR, "outputs")
REGISTRY_PATH = os.path.join(BASE_DIR, "voice_registry.json")

os.makedirs(VOICES_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

if not os.path.exists(REGISTRY_PATH):
    with open(REGISTRY_PATH, "w", encoding="utf-8") as f:
        json.dump({}, f)

LATENT_CACHE = {}
print("[OK] Voice profile storage initialized.")

In [ ]:
#@title 3. Configure FastAPI Inference Endpoints (Compatible with Vox Node.js Backend)
import uuid
import shutil
import soundfile as sf
from typing import Optional
from fastapi import FastAPI, UploadFile, File, Form, HTTPException, Header, Request
from fastapi.responses import Response

ENGINE_API_KEY = os.environ.get("VOX_ENGINE_API_KEY", "").strip()

SUPPORTED_LANGUAGES = [
    "en", "hi", "bn", "es", "fr", "de", "it", "pt", "pl",
    "tr", "ru", "nl", "cs", "ar", "zh-cn", "ja", "hu", "ko"
]

app = FastAPI(title="VOX AI ENGINE (XTTS-v2 Colab Node)", version="1.0.0")

def verify_auth(authorization: Optional[str]):
    if not ENGINE_API_KEY:
        return
    if not authorization or not authorization.startswith("Bearer "):
        raise HTTPException(status_code=401, detail="Missing Bearer token.")
    token = authorization.replace("Bearer ", "", 1).strip()
    if token != ENGINE_API_KEY:
        raise HTTPException(status_code=403, detail="Invalid inference engine token.")

def load_registry() -> dict:
    with open(REGISTRY_PATH, "r", encoding="utf-8") as f:
        return json.load(f)

def save_registry(data: dict):
    with open(REGISTRY_PATH, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2)

def compute_or_get_latents(speaker_id: str, wav_path: str):
    if speaker_id in LATENT_CACHE:
        return LATENT_CACHE[speaker_id]
    gpt_cond_latent, speaker_embedding = xtts_model.get_conditioning_latents(
        audio_path=[wav_path]
    )
    LATENT_CACHE[speaker_id] = (gpt_cond_latent, speaker_embedding)
    return gpt_cond_latent, speaker_embedding

@app.get("/health")
async def health_check(authorization: Optional[str] = Header(default=None)):
    verify_auth(authorization)
    registry = load_registry()
    return {
        "status": "ok",
        "engine": "xtts-v2",
        "device": DEVICE,
        "saved_voices": len(registry),
        "supported_languages": SUPPORTED_LANGUAGES
    }

@app.get("/api/voices")
async def list_voices(authorization: Optional[str] = Header(default=None)):
    verify_auth(authorization)
    registry = load_registry()
    voices = [{"id": k, "name": v["name"]} for k, v in registry.items()]
    return {"voices": voices}

@app.post("/api/voices/register")
async def register_voice(
    name: str = Form(...),
    speaker_wav: UploadFile = File(...),
    authorization: Optional[str] = Header(default=None)
):
    verify_auth(authorization)
    speaker_id = f"vox_{uuid.uuid4().hex[:12]}"
    wav_path = os.path.join(VOICES_DIR, f"{speaker_id}.wav")

    try:
        with open(wav_path, "wb") as buffer:
            shutil.copyfileobj(speaker_wav.file, buffer)

        compute_or_get_latents(speaker_id, wav_path)

        registry = load_registry()
        registry[speaker_id] = {
            "name": name.strip(),
            "wav_path": wav_path
        }
        save_registry(registry)

        return {
            "success": True,
            "speaker_id": speaker_id,
            "name": name.strip()
        }
    except Exception as e:
        if os.path.exists(wav_path):
            os.remove(wav_path)
        raise HTTPException(status_code=500, detail=f"Voice embedding extraction failed: {str(e)}")

@app.delete("/api/voices/{speaker_id}")
async def delete_voice(speaker_id: str, authorization: Optional[str] = Header(default=None)):
    verify_auth(authorization)
    registry = load_registry()
    entry = registry.pop(speaker_id, None)
    LATENT_CACHE.pop(speaker_id, None)
    if entry:
        wav_path = entry.get("wav_path")
        if wav_path and os.path.exists(wav_path):
            os.remove(wav_path)
        save_registry(registry)
    return {"success": True, "deleted": speaker_id}

@app.post("/api/tts")
async def synthesize_speech(request: Request, authorization: Optional[str] = Header(default=None)):
    verify_auth(authorization)
    content_type = request.headers.get("content-type", "")
    temp_wav_path = None

    try:
        if "multipart/form-data" in content_type:
            form = await request.form()
            text = str(form.get("text", "")).strip()
            language = str(form.get("language", "en")).strip()
            speaker_id = str(form.get("speaker_id", "")).strip()
            uploaded_wav = form.get("speaker_wav")

            if uploaded_wav is not None and hasattr(uploaded_wav, "file"):
                temp_wav_path = os.path.join(OUTPUT_DIR, f"ref_{uuid.uuid4().hex[:10]}.wav")
                with open(temp_wav_path, "wb") as f:
                    shutil.copyfileobj(uploaded_wav.file, f)
        else:
            payload = await request.json()
            text = str(payload.get("text", "")).strip()
            language = str(payload.get("language", "en")).strip()
            speaker_id = str(payload.get("speaker_id", "")).strip()

        if not text:
            raise HTTPException(status_code=400, detail="Text script cannot be empty.")
        if language not in SUPPORTED_LANGUAGES:
            raise HTTPException(status_code=400, detail=f"Unsupported language '{language}'. Supported: {SUPPORTED_LANGUAGES}")

        registry = load_registry()
        ref_wav = temp_wav_path

        if not ref_wav:
            if speaker_id not in registry:
                raise HTTPException(status_code=404, detail=f"Voice profile '{speaker_id}' not found in Colab session.")
            ref_wav = registry[speaker_id]["wav_path"]

        cache_key = speaker_id if (speaker_id in registry and not temp_wav_path) else f"tmp_{uuid.uuid4().hex[:8]}"
        gpt_cond_latent, speaker_embedding = compute_or_get_latents(cache_key, ref_wav)
        if cache_key.startswith("tmp_"):
            LATENT_CACHE.pop(cache_key, None)

        out = xtts_model.inference(
            text=text,
            language=language,
            gpt_cond_latent=gpt_cond_latent,
            speaker_embedding=speaker_embedding,
            temperature=0.7
        )

        out_wav_path = os.path.join(OUTPUT_DIR, f"synth_{uuid.uuid4().hex[:12]}.wav")
        sf.write(out_wav_path, out["wav"], 24000)

        with open(out_wav_path, "rb") as f:
            wav_bytes = f.read()

        os.remove(out_wav_path)
        return Response(content=wav_bytes, media_type="audio/wav")

    except HTTPException:
        raise
    except Exception as e:
        raise HTTPException(status_code=500, detail=f"XTTS inference error: {str(e)}")
    finally:
        if temp_wav_path and os.path.exists(temp_wav_path):
            os.remove(temp_wav_path)

print("[OK] FastAPI inference routes configured.")

In [ ]:
#@title 4. Start Secure HTTPS Tunnel & Live Inference Server
import threading
import time
import getpass
import uvicorn
import nest_asyncio
from pyngrok import ngrok, conf

ngrok_token = os.environ.get("NGROK_AUTHTOKEN", "").strip()
try:
    from google.colab import userdata
    if not ngrok_token:
        ngrok_token = userdata.get("NGROK_AUTHTOKEN") or ""
    if not ENGINE_API_KEY:
        secret_engine_key = userdata.get("VOX_ENGINE_API_KEY") or ""
        if secret_engine_key:
            ENGINE_API_KEY = secret_engine_key.strip()
            os.environ["VOX_ENGINE_API_KEY"] = ENGINE_API_KEY
except Exception:
    pass

if not ngrok_token:
    print("Enter your free Ngrok Authtoken from https://dashboard.ngrok.com/get-started/your-authtoken")
    ngrok_token = getpass.getpass("Ngrok Authtoken: ").strip()

if not ngrok_token:
    raise ValueError("Ngrok Authtoken is required to expose an HTTPS endpoint to Render.")

conf.get_default().auth_token = ngrok_token
ngrok.kill()

nest_asyncio.apply()
PORT = 8000

def run_uvicorn():
    uvicorn.run(app, host="0.0.0.0", port=PORT, log_level="warning")

server_thread = threading.Thread(target=run_uvicorn, daemon=True)
server_thread.start()
time.sleep(2)

tunnel = ngrok.connect(PORT, bind_tls=True)
public_https_url = tunnel.public_url

print("=" * 62)
print("   VOX AI ENGINE (XTTS-v2) IS LIVE FOR TESTING")
print("=" * 62)
print(f"HTTPS Inference URL : {public_https_url}")
print(f"Health Check URL    : {public_https_url}/health")
print(f"Auth Protection     : {'ENABLED (Bearer Token)' if ENGINE_API_KEY else 'OPEN (Set VOX_ENGINE_API_KEY in Colab Secrets to lock)'}")
print("=" * 62)
print("Copy the HTTPS Inference URL above into:")
print("  1) Render Env Var: AI_INFERENCE_URL")
print("  2) OR Android App -> Settings -> XTTS-v2 Inference Server URL")
print("\n[NOTE] Keep this Colab tab open while testing. Free sessions time out when idle.")